# Unix-terminal. Работа с оборудованием

## Мотивация

Обучение модели упирается не в код, а в железо. `batch_size` подбирают под память видеокарты, `num_workers` — под число ядер, скорость загрузки данных — под диск. Пока не знаешь, что стоит в машине, эти числа выбираются наугад, а потом обучение падает с `CUDA out of memory` или простаивает, потому что данные не успевают подъезжать.

Вторая причина — чужая машина. На сервере лаборатории или в облаке нет ни диспетчера устройств, ни системного монитора: узнать, сколько там памяти, какие диски смонтированы и занята ли видеокарта соседом, можно только из терминала. Тот же вопрос возникает, когда просят помочь с чужой ошибкой: первый шаг — выяснить, на чём всё это запущено.

Ключевая мысль занятия: в Linux **железо видно через файлы**. Не нужны специальные библиотеки и утилиты — почти всё читается обычным `cat` из `/proc` и `/sys`, а знакомые `lscpu` и `free` просто приводят эти файлы в человеческий вид.

> **Как устроен семинар.** Ноутбук состоит из двух частей: **демонстрация**
> (разделы 1–7) — её показывают на занятии и повторяют за преподавателем, и
> **справочник** (разделы 8–10) — его не демонстрируют, он нужен при решении
> задач.
>
> Свёрнутые блоки **🎙 Заметка преподавателя** — это то, что рассказывается
> вслух на занятии. Разворачивайте их потом, при подготовке к защите.
>
> Всё, что вы увидите дальше, читается прямо с вашей машины: заранее
> подготовленных файлов с выводом здесь нет, поэтому числа у каждого будут
> свои. Работаем в домашнем каталоге (`~/seminar-06/`), `/tmp` не используем.

## 1. Кто я и на какой машине

Первые четыре команды на любой незнакомой машине. Они отвечают на вопросы «под кем я работаю», «где нахожусь», «какое ядро» и «какой дистрибутив» — с этого начинается любой разговор в чате поддержки.

In [ ]:
%%bash
whoami       # под каким пользователем выполняются команды
pwd          # в каком каталоге мы сейчас
uname -srmo  # ядро, его версия и архитектура процессора

Версия ядра и версия дистрибутива — разные вещи. Ядро печатает `uname`, а название системы лежит текстовым файлом в `/etc/os-release`: его читают и скрипты, и установщики пакетов.

In [ ]:
%%bash
# Файл состоит из строк вида КЛЮЧ="значение" — его удобно и читать, и разбирать.
grep -E '^(NAME|VERSION|ID)=' /etc/os-release

#### ❓ **Вопрос**: `uname -r` печатает `6.8.0-136-generic`, а в `/etc/os-release` написано `Ubuntu 22.04`. Это противоречие?

<details>

<summary><strong>Ответ</strong></summary>

Нет. `uname` показывает версию **ядра**, `/etc/os-release` — версию **дистрибутива**, то есть набора программ и библиотек вокруг ядра. Ядро обновляется отдельно и часто оказывается новее того, с которым система вышла. Для вопросов «поддерживается ли моя видеокарта» важно первое, для «каким пакетным менеджером ставить» — второе.

</details>

## 2. `/proc` и `/sys`: железо как файлы

`/proc` и `/sys` — псевдофайловые системы. На диске их нет: содержимое генерирует ядро в момент чтения. Поэтому `ls -l` показывает у таких файлов нулевой размер, хотя `cat` печатает текст.

- `/proc/cpuinfo`, `/proc/meminfo` — процессор и память;
- `/proc/loadavg` — средняя загрузка за 1, 5 и 15 минут;
- `/proc/PID/` — сведения о конкретном процессе;
- `/sys/block/` — блочные устройства;
- `/sys/class/thermal/` — датчики температуры.

Это и есть главный интерфейс к железу: `lscpu`, `free`, `top` читают ровно эти файлы.

In [ ]:
%%bash
ls -l /proc/meminfo /proc/loadavg   # размер 0: содержимое сочиняется в момент чтения

In [ ]:
%%bash
# А содержимое есть: три числа загрузки, счётчики процессов и последний выданный pid.
cat /proc/loadavg

#### ❓ **Вопрос**: `ls -l /proc/meminfo` показывает размер 0, но `cat /proc/meminfo` печатает десятки строк. Как это возможно?

<details>

<summary><strong>Ответ</strong></summary>

`/proc` не хранит файлы на диске: это интерфейс к ядру, оформленный как файловая система. Размер заранее неизвестен, потому что содержимое формируется в момент чтения, и ядро сообщает 0. Читать такие файлы нужно целиком, а не полагаться на их размер.

</details>

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Идея «всё есть файл» — из Unix, но `/proc` придумали позже, в Bell Labs для
Plan 9, и уже оттуда он приехал в Linux. Прелесть решения в том, что для
чтения состояния ядра не нужно ни системных вызовов, ни специальных библиотек:
работают те же `cat`, `grep` и `open()`, что и для обычных файлов.

Побочный эффект — почти все утилиты мониторинга это тонкие обёртки над
`/proc`. `htop`, `free`, `ps` читают ровно те же файлы, что и вы сейчас. Когда
на сервере нет нужной утилиты, а поставить нельзя, это спасает.

</details>

## 3. Процессор: читаем `/proc/cpuinfo`

`/proc/cpuinfo` — список блоков, по одному на каждый **логический** процессор. Внутри блока важны четыре строки:

- `processor` — порядковый номер логического процессора;
- `model name` — модель, одинаковая во всех блоках;
- `physical id` — номер сокета (физического процессора);
- `core id` — номер ядра внутри сокета.

Отсюда правило чтения: **сколько блоков — столько логических процессоров**, а число физических ядер — это количество уникальных пар `physical id` + `core id`.

In [ ]:
%%bash
# Один блок целиком: посмотрим, из каких строк он состоит.
head -n 12 /proc/cpuinfo

In [ ]:
%%bash
grep -c '^processor' /proc/cpuinfo     # блоков в файле — столько логических процессоров
grep -m 1 '^model name' /proc/cpuinfo  # -m 1: модель во всех блоках одна и та же

Разница между ядрами и логическими процессорами возникает из-за SMT (у Intel это Hyper-Threading): одно физическое ядро показывает системе два потока.

Сокеты, ядра и потоки

In [ ]:
%%bash
# paste - - склеивает соседние строки попарно, sort -u оставляет уникальные пары:
# сколько уникальных пар «сокет + ядро», столько физических ядер.
grep -E '^(physical id|core id)' /proc/cpuinfo | paste - - | sort -u | wc -l

Те же данные в готовом виде показывает `lscpu`, а `nproc` печатает одно число — сколько логических процессоров доступно программе. Именно его подставляют в `num_workers` и в `-j` у сборщиков.

In [ ]:
%%bash
lscpu | grep -E '^(Architecture|Model name|CPU\(s\)|Thread|Core|Socket)'
nproc

#### ❓ **Вопрос**: Пусть `lscpu` показывает `Socket(s): 1` и `Core(s) per socket: 6`, а `nproc` печатает 12. Откуда взялись ещё шесть процессоров и какое число брать для `num_workers`?

<details>

<summary><strong>Ответ</strong></summary>

Ответ в строке `Thread(s) per core: 2` — это SMT: каждое из 6 физических ядер показывает системе два логических процессора, `1 × 6 × 2 = 12`. Два потока делят исполнительные блоки одного ядра, поэтому на счётной нагрузке они не дают двукратного ускорения. Для загрузчиков данных обычно отталкиваются от числа физических ядер и уточняют замерами.

</details>

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Про SMT стоит проговорить, откуда берётся путаница в магазине. На коробке
пишут «6 ядер, 12 потоков», и покупатель уверен, что купил двенадцать
процессоров. На самом деле второй поток использует простои первого: пока одно
вычисление ждёт данные из памяти, ядро исполняет другое. На счётной нагрузке,
где данные всегда в кэше, прирост может быть нулевым, а иногда и отрицательным
— потоки дерутся за кэш.

Отсюда практическое правило для обучения: `num_workers` по числу физических
ядер, а дальше замерять. Ровно так же в HPC-кластерах SMT часто просто
выключают в BIOS.

</details>

## 4. Память: `/proc/meminfo` и `free`

`/proc/meminfo` — те же числа в килобайтах, `free -h` — они же в человеческом виде.

Ключевая колонка — не `free`, а `available`: это сколько памяти реально можно занять новой программе. Разница возникает из-за страничного кэша: ядро держит в памяти прочитанные с диска данные, но отдаст их по первому требованию.

Из чего складывается вывод free -h

In [ ]:
%%bash
grep -E '^(MemTotal|MemFree|MemAvailable|SwapTotal)' /proc/meminfo

In [ ]:
%%bash
free -h   # то же самое, но с единицами измерения и с колонкой buff/cache

`swap` — область на диске, куда ядро вытесняет страницы, когда память кончается. Обучение, уехавшее в swap, не падает, а становится в сотни раз медленнее — по графику загрузки это выглядит как «всё зависло».

#### ❓ **Вопрос**: В выводе `free -h` колонка `free` почти пустая, а в `available` остаются гигабайты. Пора ли закрывать программы?

<details>

<summary><strong>Ответ</strong></summary>

Нет. Разницу между этими колонками занимает страничный кэш: ядро держит в памяти прочитанные с диска данные и отдаст их, как только память понадобится программе. Ориентироваться нужно на `available`. Тревожный признак другой — `available` близко к нулю и растёт использование swap.

</details>

## 5. Устройства: блочные и символьные

Железо в Linux представлено файлами в `/dev`. Первая буква в выводе `ls -l` говорит, какого рода это устройство:

- **`b`** — **блочное**: данные читаются и пишутся блоками, есть произвольный доступ и кэш. Это диски и разделы;
- **`c`** — **символьное**: поток байт без произвольного доступа. Это терминалы, звук, `/dev/null`, `/dev/random`.

Вместо размера у таких файлов стоят два числа — **major** и **minor**: major говорит ядру, какой драйвер обслуживает устройство, minor — какое именно из однотипных устройств.

In [ ]:
%%bash
# Первая колонка: b — блочное устройство, c — символьное.
ls -l /dev/null /dev/random /dev/sda 2>/dev/null || ls -l /dev/null /dev/random

In [ ]:
%%bash
# major:minor вместо размера — по ним ядро находит нужный драйвер.
lsblk -a -o NAME,MAJ:MIN,TYPE,SIZE | head -n 6

#### ❓ **Вопрос**: Почему у файла `/dev/sda` вместо размера напечатаны два числа, например `8, 0`?

<details>

<summary><strong>Ответ</strong></summary>

Это major и minor. `/dev/sda` — не файл с данными, а точка входа в драйвер: major (8) говорит ядру, какой драйвер обслуживает устройство, minor (0) — какое именно устройство этого драйвера имеется в виду. Размер диска берут не отсюда, а из `lsblk` или `/sys/block/sda/size`.

</details>

### Диски и файловые системы

Теперь от устройств к тому, что на них лежит. `lsblk` показывает блочные устройства деревом: диск → разделы → точки монтирования.

Флаг `-e 7` исключает устройства с major-номером 7 — это loop-устройства. За ними стоят не диски, а файлы-образы: каждый пакет snap монтируется как отдельная read-only файловая система.

In [ ]:
%%bash
lsblk -e 7 -o NAME,SIZE,TYPE,FSTYPE,MOUNTPOINTS

In [ ]:
%%bash
df -h .                                # сколько занято там, где мы сейчас находимся
findmnt -n -o SOURCE,TARGET,FSTYPE /   # какое устройство смонтировано в корень

`df` спрашивает файловую систему и отвечает мгновенно; `du -sh КАТАЛОГ` обходит дерево и на большом каталоге работает долго.

Разметку дисков смотрят и меняют `fdisk -l` и `parted -l`, а подключают файловые системы `mount` и `umount`:

```bash
sudo fdisk -l /dev/nvme0n1
sudo mount /dev/sdb1 /mnt/data
sudo umount /mnt/data
```

Всё это требует прав root, и здесь мы такие команды не запускаем. Список разделов читать безопасно, а вот `fdisk` в интерактивном режиме меняет таблицу разделов — ошибка стоит данных всего диска. Постоянные точки монтирования описывают в `/etc/fstab`; опечатка в нём может привести к тому, что машина не загрузится.

#### ❓ **Вопрос**: Зачем в `lsblk` добавляют `-e 7`, если loop-устройства тоже настоящие устройства ядра?

<details>

<summary><strong>Ответ</strong></summary>

Loop-устройство подставляет обычный файл вместо блочного устройства. В Ubuntu так смонтирован каждый пакет snap, поэтому их десятки, все read-only и заполнены на 100 %. К физическим дискам они отношения не имеют, и в отчёте о железе только мешают.

</details>

## 6. Шины и видеокарта

`lspci` перечисляет устройства на шине PCI: контроллеры дисков, сетевые карты, видеокарты. `lsusb` делает то же для USB.

Строка `Kernel driver in use` в выводе `lspci -k` отвечает на самый частый вопрос: устройство видно, но не работает — драйвер вообще загружен?

In [ ]:
%%bash
lspci | wc -l       # сколько всего устройств на шине PCI
lsusb | head -n 3   # первые три устройства USB, включая корневые хабы

In [ ]:
%%bash
# -k добавляет строку Kernel driver in use — кто на самом деле управляет картой.
lspci -k | grep -A 3 -i 'VGA compatible'

`nvidia-smi` показывает видеокарты NVIDIA: модель, занятую и общую память, загрузку и процессы, которые их держат. Утилита ставится вместе с драйвером, поэтому на машине без такой карты её просто нет.

Для скриптов обычный вывод не годится — это таблица для человека. Есть машинный режим: `--query-gpu` перечисляет нужные поля, `--format=csv,noheader` убирает оформление, `nounits` — единицы измерения. На сервере с четырьмя картами

```bash
nvidia-smi --query-gpu=index,name,memory.total,memory.used --format=csv,noheader
```

выведет

```
0, NVIDIA A100-SXM4-40GB, 40960 MiB, 39012 MiB
1, NVIDIA A100-SXM4-40GB, 40960 MiB, 1024 MiB
```

Свободную память считают как разницу `memory.total` и `memory.used`.

In [ ]:
%%bash
# Утилита ставится вместе с драйвером NVIDIA: нет карты — нет и команды.
nvidia-smi || echo 'nvidia-smi недоступна: нет карт NVIDIA или не установлен драйвер'

#### ❓ **Вопрос**: Почему в скрипте берут `--format=csv,noheader,nounits`, а не разбирают обычную таблицу `nvidia-smi`?

<details>

<summary><strong>Ответ</strong></summary>

Обычный вывод — это отчёт для человека: рамки, объединённые ячейки, единицы измерения внутри значений. Его разметка не обещана стабильной и меняется с версией драйвера. Режим `--query-gpu` возвращает ровно запрошенные поля, `noheader` и `nounits` убирают всё, что пришлось бы вырезать вручную, и остаётся CSV из чисел.

</details>

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Про драйверы NVIDIA полезно предупредить заранее, потому что на этом теряют
часы. Драйвер живёт на хосте, библиотеки CUDA — внутри окружения или образа, и
версии должны быть совместимы: образ с CUDA 12.4 не заработает на сервере с
драйвером под 11.8. Сообщение при этом будет невнятное, вроде «no kernel image
is available for execution».

Отсюда же правило для контейнеров: драйвер в образ не кладут никогда, его
пробрасывают с хоста через `nvidia-container-toolkit`.

</details>

## 7. Кто нагружает машину: `top` и `htop`

`top` показывает процессы в реальном времени и сортирует их по загрузке процессора. Внутри: `M` — сортировать по памяти, `P` — по процессору, `1` — показать ядра по отдельности, `q` — выход.

`htop` — то же самое, но нагляднее: цветные шкалы, дерево процессов, поиск по `F3`. Ставится отдельно (`sudo apt install htop`).

В ноутбуке интерактивный режим не работает, поэтому запускаем `top` в пакетном режиме: `-b` — вывод в поток, `-n 1` — один снимок.

In [ ]:
%%bash
# Шапка top: время работы, средняя загрузка, счётчики задач, процессор и память.
top -b -n 1 | head -n 5

In [ ]:
%%bash
# -o %MEM сортирует по памяти: так ищут, кто съел всю оперативку.
top -b -n 1 -o %MEM | head -n 12 | tail -n 6

#### ❓ **Вопрос**: В шапке `top` написано `load average: 12.0, 11.8, 9.4`, а процессоров в машине 12. Это много?

<details>

<summary><strong>Ответ</strong></summary>

Само по себе — нет: средняя загрузка примерно равна числу процессоров означает, что машина занята полностью, но очереди не растёт. Тревожно, когда число заметно больше количества процессоров — задачи начали ждать. Важная оговорка для Linux: в load average попадают и процессы, ждущие диск, поэтому высокая загрузка при простое процессора обычно означает, что упёрлись в диск.

</details>

---

# Справочная часть

Дальше — то, что не показывают на занятии: пригодится при решении задач и при
подготовке к защите.

## 8. Здоровье и паспорт железа

Часть сведений доступна только root, потому что читается напрямую с устройств.

- `sudo smartctl -a /dev/sda` — данные SMART: часы работы, число включений, переназначенные секторы, температура. Ставится пакетом `smartmontools`. Это первое, что смотрят, когда диск начал «подтормаживать»: рост `Reallocated_Sector_Ct` означает, что диск сыплется и его пора менять;
- `sudo smartctl -H /dev/sda` — короткий вердикт `PASSED` или `FAILED`;
- `sudo dmidecode -t memory` — паспортные данные плашек памяти из таблиц BIOS: объём, тип, частота, свободные слоты. Так узнают, можно ли доставить память, не открывая корпус;
- `sudo hdparm -t /dev/sda` — грубый замер скорости чтения;
- `sensors` (пакет `lm-sensors`) — температуры и обороты вентиляторов.

In [ ]:
%%bash
# Проверяем, что установлено: на учебной машине smartctl обычно нет.
command -v smartctl || echo 'smartctl не установлен: sudo apt install smartmontools'
command -v sensors  || echo 'sensors не установлен: sudo apt install lm-sensors'

## 9. Вызов утилит из Python: `subprocess`

Разовый вопрос закрывается командой в терминале, а «проверь перед запуском, что на GPU хватает памяти» — это уже скрипт. Те же утилиты вызывают из программы через `subprocess.run`:

- команда передаётся **списком аргументов**, а не строкой;
- `capture_output=True` забирает stdout и stderr вместо вывода на экран;
- `text=True` возвращает строки, а не байты;
- `check=True` бросает исключение, если команда завершилась ненулевым кодом.

In [ ]:
import subprocess

In [ ]:
# capture_output забирает вывод в переменную, text даёт строку вместо байтов,
# check=True превращает ненулевой код команды в исключение.
result = subprocess.run(["nproc"], capture_output=True, text=True, check=True)
print(result.stdout.strip(), "| код:", result.returncode)

Если программы нет в системе, `subprocess` бросает `FileNotFoundError` — это нормальный сценарий для `nvidia-smi`, и его обрабатывают, а не считают аварией.

In [ ]:
# Возвращает вывод команды либо объяснение, почему его нет:
# отсутствие nvidia-smi — обычный сценарий, а не авария.
def run(command):
    try:
        return subprocess.run(command, capture_output=True, text=True, check=True).stdout.strip()
    except FileNotFoundError as error:
        return f"нет такой программы: {error.filename}"

In [ ]:
# Первая команда есть, второй нет — обе обработаны без падения программы.
print(run(["nproc"]), "|", run(["nvidia-smi"]))

Списком аргументов команду передают не для красоты. С `shell=True` строку разбирает оболочка, и любой символ вроде `;` или `|` внутри подставленного значения станет частью команды.

In [ ]:
# Представим, что имя файла пришло от пользователя и содержит точку с запятой.
name = "cpuinfo; echo ВЫПОЛНИЛАСЬ ЧУЖАЯ КОМАНДА"
# shell=True отдаёт строку оболочке — она увидит здесь ДВЕ команды и выполнит обе.
print(subprocess.run(f"head -1 /proc/{name}", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# Со списком аргументов всё значение уходит одним аргументом: команда просто не нашла файл.
print(subprocess.run(["head", "-1", f"/proc/{name}"], capture_output=True, text=True).stderr)

#### ❓ **Вопрос**: Скрипт получает имя файла от пользователя и вызывает `subprocess.run(f"cat /proc/{name}", shell=True)`. Что произойдёт, если в имени окажется `cpuinfo; rm -rf ~`?

<details>

<summary><strong>Ответ</strong></summary>

Оболочка увидит две команды, разделённые `;`, и выполнит обе — вторая удалит домашний каталог. Со списком аргументов такого не бывает: вся строка целиком уходит одним аргументом, программа просто не находит файл с таким именем и возвращает ошибку. Поэтому `shell=True` не используют с данными, пришедшими извне.

</details>

## 10. `fork`: как вообще появляются процессы

`subprocess` — удобная обёртка, но под ней лежит механизм самой системы.

`fork()` создаёт копию текущего процесса. После вызова выполняются оба: у ребёнка `fork()` возвращает `0`, у родителя — pid ребёнка. Дальше ребёнок обычно вызывает `exec()` и заменяет свой образ на другую программу — так запускается любая команда в оболочке. Родитель обязан забрать результат через `wait()`: пока он этого не сделал, завершившийся ребёнок остаётся в таблице процессов зомби (`Z` в выводе `ps`).

fork, exec и wait

In [ ]:
%%bash
# fork вернул 0 ребёнку и pid ребёнка родителю — печатают оба, ждёт только родитель.
python3 -c "import os; pid = os.fork(); print('дочерний' if pid == 0 else 'родитель', os.getpid(), flush=True); pid and os.wait()"

Запускаем отдельным процессом, а не в ядре ноутбука: форкать сам Jupyter — плохая идея, дочерний процесс унаследует его соединение с браузером.

В следующей ячейке `print` выполняется ровно один раз — до `fork`.

In [ ]:
%%bash
# print один, а строк в выводе будет две: буфер скопировался вместе с процессом.
python3 -c "import os; print('до fork'); os.fork() and os.wait()"

#### ❓ **Вопрос**: Строка `print("до fork")` выполняется один раз, а в выводе она появилась дважды. Почему?

<details>

<summary><strong>Ответ</strong></summary>

Вывод Python буферизуется, и когда он идёт не в терминал, а в пайп, буфер сбрасывается только при завершении программы. К моменту `fork()` строка ещё лежит в буфере, а `fork` копирует процесс вместе с ним — теперь буфер есть у обоих. При завершении каждый сбрасывает свою копию, и строка печатается дважды. Лечится вызовом `flush=True` перед `fork()`; по той же причине в дочернем процессе используют `os._exit()`, который буферы не трогает.

</details>

## Дополнительно

### `nproc` внутри контейнера

В контейнере и под управлением планировщика задач `nproc` может показать все процессоры хоста, хотя процессу разрешена лишь часть. Ограничение задаётся через cgroups и в это число не попадает.

Сколько процессоров реально доступно процессу, показывает маска привязки:

```python
len(os.sched_getaffinity(0))
```

Ошибка встречается часто: `num_workers` выставляют по `os.cpu_count()`, получают вдвое больше потоков, чем разрешено, и обучение замедляется вместо ускорения.

### Наблюдение за нагрузкой

- `uptime` и `/proc/loadavg` — средняя загрузка за 1, 5 и 15 минут;
- `vmstat 1` — память, swap и процессор раз в секунду;
- `iostat -x 1` (пакет `sysstat`) — нагрузка на диски, колонка `%util`;
- `nvtop` — то же, что `htop`, но для видеокарт;
- `watch -n 1 КОМАНДА` — повторять любую команду и обновлять экран.

Если загрузка процессора низкая, а обучение идёт медленно, узкое место обычно в диске или в загрузчике данных, а не в модели.